In [42]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 12
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

Daniel Lei
Turki 
Ruoxi
Grace

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [43]:

tickers_df = pd.read_csv('Tickers_Example.csv', header = None, dtype=str)

start_date = '2025-10-01'
end_date = '2026-10-01'

ticker_list = []
for ticker in tickers_df.iloc[:, 0].dropna():
    ticker = ticker.replace(' ', '').upper()
    # skip blanks any title rows and duplicates (e.g. RY.TO listed twice)
    if ticker == '' or ticker == 'TICKER' or ticker in ticker_list:
        continue

    ticker_list.append(ticker)

In [44]:
symbols = ticker_list + ['^GSPC', 'CAD=X']

close = pd.DataFrame()
volume = pd.DataFrame()
for attempt in range(3):
    missing = [t for t in symbols if t not in close.columns]
    if len(missing) == 0:
        break
    data = yf.download(missing, start=start_date, end=end_date, auto_adjust=False, progress=False)
    if data.empty:
        continue
    new_close = data['Close'].dropna(axis=1, how='all')
    close = pd.concat([close, new_close], axis=1)
    volume = pd.concat([volume, data['Volume'][new_close.columns]], axis=1)

print('No price data (invalid or delisted):', [t for t in ticker_list if t not in close.columns]) #i asked jester abt deleting invalid tickers and i think this is a good way to approach solving it.

$CELG: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$MON: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found

5 Failed downloads:
['CELG', 'RTN', 'MON', 'AGN']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$MON: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)

5 Failed downloads:
['MON', 'CELG', 'AGN', 'RTN']: possibly delisted; no timezone found
['BK']: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$MON: possibly delisted; no timezone found
$CELG: possibly delisted; no timezone found
$AGN: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone fou

No price data (invalid or delisted): ['AGN', 'BK', 'CELG', 'MON', 'RTN']


In [45]:
stock_volume = volume.drop(columns=['^GSPC', 'CAD=X'], errors='ignore')

# months with fewer than 18 trading days are ignored
days_per_month = stock_volume.groupby(stock_volume.index.to_period('M')).transform('count')
stock_volume = stock_volume.where(days_per_month >= 18)

mean_volumes = stock_volume.mean()
filtered_tickers = mean_volumes[mean_volumes >= 5000].index.tolist()
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [46]:
allowed_exchanges = ['NMS', 'NGM', 'NCM', 'NYQ', 'NYS', 'ASE', 'PCX', 'BTS', 'BATS', 'NAS', 'TOR', 'VAN', 'CNQ', 'NEO']
#got rid of OTC
info_stocks = {}
for ticker in filtered_tickers:
    info = yf.Ticker(ticker).get_info()
    if info.get('quoteType') != 'EQUITY':
        print(f'Removed (not a stock): {ticker}')
    elif info.get('currency') not in ['CAD', 'USD']:
        print(f'Removed (currency is {info.get("currency")}): {ticker}')
    elif info.get('exchange') not in allowed_exchanges:
        print(f'Removed (not listed in Canada/US, exchange is {info.get("exchange")}): {ticker}')
    else:
        info_stocks[ticker] = {'Name': info.get('shortName', ticker),
                               'Sector': info.get('sector') or 'Unknown',
                               'Currency': info['currency']}

info_stocks = pd.DataFrame(info_stocks).T
filtered_tickers = info_stocks.index.tolist()
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [47]:
# goal here is if the same company appears twice (e.g. SHOP and SHOP.TO) u keep only one listing
seen = {}   

# loop over a copy cos we remove items from filtered_tickers inside the loop
for t in filtered_tickers.copy():
    # clean the company name so small differences don't matter ("Shopify Inc." vs "shopify inc")
    key = info_stocks.loc[t, 'Name'].lower().replace('.', '').replace(',', '').strip()
    if key not in seen: 
        seen[key] = t
    elif mean_volumes[t] > mean_volumes[seen[key]]:
        print(f'Removed (second listing of the same company): {seen[key]}')
        filtered_tickers.remove(seen[key])
        seen[key] = t
    else:
        # company already seen, and the one we kept trades more so remove this one
        print(f'Removed (second listing of the same company): {t}')
        filtered_tickers.remove(t)

info_stocks = info_stocks.loc[filtered_tickers]
filtered_tickers

['AAPL',
 'ABBV',
 'ABT',
 'ACN',
 'AIG',
 'AMZN',
 'AXP',
 'BA',
 'BAC',
 'BB.TO',
 'BIIB',
 'BLK',
 'BMY',
 'C',
 'CAT',
 'CL',
 'KO',
 'LLY',
 'LMT',
 'MO',
 'MRK',
 'PEP',
 'PFE',
 'PM',
 'PYPL',
 'QCOM',
 'RY.TO',
 'SHOP.TO',
 'T.TO',
 'TD.TO',
 'TXN',
 'UNH',
 'UNP',
 'UPS',
 'USB']

In [58]:
# CAD per 1 USD, each day. ffill() copies the last known value forward over gaps
# (e.g. a US holiday when the Canadian market was open)
fx = close['CAD=X'].ffill()


market = close['^GSPC'].ffill() * fx
prices = close[filtered_tickers].ffill()

# convert USD to CAD 
for t in filtered_tickers:
    if info_stocks.loc[t, 'Currency'] == 'USD':
        prices[t] = prices[t] * fx
# daily return = today's price / yesterday's price - 1

returns = prices.pct_change().iloc[1:]
market_returns = market.pct_change().iloc[1:]#drops the first row, which is empty because it has no "yesterday"
# beta = cov(stock, market) / var(market)
beta = {}
for t in filtered_tickers:
    beta[t] = returns[t].cov(market_returns) / market_returns.var()
beta = pd.Series(beta)
standard_deviation = returns.std()
average_return = returns.mean()
pd.DataFrame({'Beta': beta, 'Daily std': standard_deviation, 'Avg daily return': average_return})

,Beta,Daily std,Avg daily return
AAPL,0.736774,0.015533,0.001223
ABBV,0.052005,0.016383,0.000471
ABT,0.132084,0.016980,-0.000947
ACN,0.337659,0.028184,-0.000625
AIG,0.102999,0.015198,-0.000074
AMZN,1.367508,0.021372,0.000768
AXP,0.998835,0.016921,-0.000082
BA,1.293161,0.021654,-0.000258
BAC,0.761664,0.014034,0.000449
BB.TO,1.406293,0.037817,0.003238


In [64]:
# turns raw scores into weights with no stock above ~15% and no sector above ~40%
# (caps set slightly under the limits so rounding cant pushthr weight over)
def apply_caps(raw, sectors, max_stock=0.1499, max_sector=0.399):
    w = raw / raw.sum()                    # start: weights proportional to raw, sum to 1
    frozen = pd.Series(False, index=w.index)     # stocks already at a cap, so they stop growing
    for i in range(100):
        over = w > max_stock + 1e-12
        w[over] = max_stock                     
        frozen = frozen | over
        for sector in sectors.unique():
            members = sectors[sectors == sector].index
            total = w[members].sum()
            if total > max_sector + 1e-12#shrink an oversized sector proportionally
                w[members] = w[members] * max_sector / total
                frozen[members] = True
        extra = 1 - w.sum()                    
        if extra < 1e-12:                       
            break
        free = ~frozen
        if not free.any():          
            break
        w[free] = w[free] + extra * raw[free] / raw[free].sum()  
    return w

SyntaxError: invalid syntax (3289506302.py, line 13)

In [59]:
buy = yf.download(filtered_tickers + ['CAD=X'], start='2026-11-16', end='2026-11-28', auto_adjust=False, progress=False)['Close'].ffill()
buy_prices = buy.iloc[-1]
fx_buy = buy_prices['CAD=X']
print('Purchase prices are from', buy.index[-1].date())

def build_portfolio(weights, budget):
    price_cad = buy_prices[weights.index].copy()
    for t in weights.index:
        if info_stocks.loc[t, 'Currency'] == 'USD':
            price_cad[t] = price_cad[t] * fx_buy

    # im thinking the fees come out of the budget so we search for the largest amount to invest
    # such that purchases + fees still fit (halve the range 100 times)
    low = 0
    high = budget
    for i in range(100):
        invest = (low + high) / 2
        shares = weights * invest / price_cad
        # fee per stock = smaller of US$2.15 or US$0.001 per share, converted to CAD
        fees = (shares * 0.001).clip(upper=2.15).sum() * fx_buy
        if invest + fees <= budget - 0.01:     
            low = invest
        else:                                  
            high = invest
    shares = weights * low / price_cad
    fees = (shares * 0.001).clip(upper=2.15).sum() * fx_buy
    value = shares * price_cad
    # output table in correct format (Price is in the stock's own currency). messed up on that one last time lol
    table = pd.DataFrame({'Ticker': weights.index,
                          'Price': buy_prices[weights.index].values,
                          'Currency': info_stocks.loc[weights.index, 'Currency'].values,
                          'Shares': shares.values,
                          'Value': value.values,
                          'Weight': (value / value.sum()).values})
    table.index = range(1, len(table) + 1)    
    return table, fees

# buy on day one, never trade again: what would the return and risk have been over the past year?
def history_stats(weights):
    units = weights / prices[weights.index].iloc[0] # units bought w/ $1 of budget
    value = (prices[weights.index] * units).sum(axis=1) #port value per day
    daily = value.pct_change().dropna()
    return {'Total return': value.iloc[-1] / value.iloc[0] - 1,
            'Daily std': daily.std(),
            'Annualized std': daily.std() * 252 ** 0.5}#252vvalid trading days in a year

# our weights vers the same stocks at equal weights
def comparison(weights):
    equal = pd.Series(1 / len(weights), index=weights.index)
    return pd.DataFrame({'Chosen weights': history_stats(weights), 'Equal weights': history_stats(equal)}).T

Purchase prices are from 2026-11-23


## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

In [65]:
# price a month ago / price at the start - 1
past_return = prices.iloc[-21] / prices.iloc[0] - 1
def z(s):
    return (s - s.mean()) / s.std()

# higher score = high beta AND strong past return
score_a = z(beta) + z(past_return)

#go down the list from best score to worst 
chosen_a = []
sector_count = {}                                    
for t in score_a.sort_values(ascending=False).index:
    sector = info_stocks.loc[t, 'Sector']
    if sector_count.get(sector, 0) >= 4: #skip this stoc if alr has 4 in the same sector
        continue
    chosen_a.append(t)
    sector_count[sector] = sector_count.get(sector, 0) + 1
    if len(chosen_a) == 12:                        
        break
chosen_a

['CAT',
 'BB.TO',
 'TXN',
 'QCOM',
 'C',
 'MRK',
 'TD.TO',
 'AMZN',
 'BIIB',
 'RY.TO',
 'SHOP.TO',
 'USB']

In [61]:
# weights prop to score (shifted so the lowest chosen score is still positive) then capsit
raw_a = score_a[chosen_a] - score_a[chosen_a].min() + 1
weights_a = apply_caps(raw_a, info_stocks.loc[chosen_a, 'Sector'])
Portfolio_Final_A, fees_a = build_portfolio(weights_a, 1000000)
display(Portfolio_Final_A)

,Ticker,Price,Currency,Shares,Value,Weight
1,CAT,510.450012,USD,204.965329,149898.734046,0.149900
2,BB.TO,11.100000,CAD,12740.090665,141415.011237,0.141416
3,TXN,263.750000,USD,306.009247,115635.547307,0.115637
4,QCOM,149.940002,USD,448.880699,96430.138014,0.096431
5,C,110.620003,USD,605.885401,96025.924096,0.096027
6,MRK,133.720001,USD,420.562233,80573.268993,0.080574
7,TD.TO,162.050003,CAD,432.902283,70151.816244,0.070152
8,AMZN,246.820007,USD,178.579081,63150.281061,0.063151
9,BIIB,249.589996,USD,136.623720,48855.968034,0.048856
10,RY.TO,263.049988,CAD,177.400128,46665.101633,0.046665


### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

The client A is the one who wishes to maximize his gains, and thye are willing to accept high levels of risks. However, the expected returns cannot be seen in the real world, and that is why all the possible stocks were graded on two measurable criteria and their z-scores were summed up, with equal weights. The first criteria is **beta**, and that means a stock that has a higher value than 1 should be volatile in its movements, and according to the CAPM theory, he will be rewarded for taking higher market risks. The second metric is **past-year returns, excluding the latest month**: stocks with good performance in the last year have been doing even better for some time (Jegadeesh and Titman, 1993), and we exclude the latest month since the winners in that period reverse their trend. We selected the top 12 stocks ranked by scores, with at most 4 stocks per sector so that any one sector does not dominate the rest. We deliberately made our portfolio small since the competition only rewards the best five days' performance, and thus a concentrated portfolio can generate a wide range of results. The higher-ranked stocks get a larger weighting (proportional to their rank, offset such that the minimum positive weight is assigned), within 15% single-stock and 40% sector caps.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

From October 1, 2025 to September 30, 2026, expressed in CAD terms, our weights produced about 49.9% compared to 42.3% on 12 same stocks equally weighted. Moreover, our portfolio had a higher level of risk: its daily standard deviation was 1.44% compared to 1.21% (annualized 22.8% compared to 19.2%). It is the type of the trade-off we look for from the growth client – more return, more risk. This proves that using the scores helped us weight the portfolio in such a way that put more money into stocks that performed the best. The above information is based on in-sample data since we use the stock selection based on the return from this period of time.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

In case the market falls during Nov 30 to Dec 4, then stocks with high betas will decline further than the market, meaning that the beta tilt will work against us. Momentum may suddenly reverse, and within five days luck plays a bigger role than any signal does. Our investments are highly volatile: BB.TO is the most volatile stock in the investment universe, but has the second-largest weight in the portfolio, which means that one bad week in this stock will affect the entire portfolio. Our investments such as the banks and MRK are selected based on strong performance in the past, but not because of high betas.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

In [53]:
# keep the calmest 60% of stocks, then add them one at a time: always the one least correlated with those already chosen
calm = standard_deviation.sort_values().index[:len(standard_deviation) * 6 // 10]
corr = returns[calm].corr()

chosen_b = [calm[0]]
sector_count = {info_stocks.loc[calm[0], 'Sector']: 1}
while len(chosen_b) < 18:
    best = None
    best_corr = 2
    for t in calm:
        sector = info_stocks.loc[t, 'Sector']
        if t in chosen_b or sector_count.get(sector, 0) >= 7:
            continue
        avg_corr = corr.loc[t, chosen_b].mean()
        if avg_corr < best_corr:
            best = t
            best_corr = avg_corr
    if best is None:
        break
    chosen_b.append(best)
    sector = info_stocks.loc[best, 'Sector']
    sector_count[sector] = sector_count.get(sector, 0) + 1
chosen_b

['RY.TO',
 'MO',
 'T.TO',
 'LMT',
 'AIG',
 'AXP',
 'ABBV',
 'AAPL',
 'UNP',
 'ABT',
 'C',
 'PM',
 'PFE',
 'TD.TO',
 'KO',
 'PEP',
 'BLK',
 'BMY']

In [62]:
# calmer stocks get more money (1 / standard deviation then caps
raw_b = 1 / standard_deviation[chosen_b]
weights_b = apply_caps(raw_b, info_stocks.loc[chosen_b, 'Sector'])
Portfolio_Final_B, fees_b = build_portfolio(weights_b, 750000)
display(Portfolio_Final_B)

,Ticker,Price,Currency,Shares,Value,Weight
1,RY.TO,263.049988,CAD,247.781338,65178.878014,0.086906
2,MO,74.870003,USD,341.417509,36623.342657,0.048832
3,T.TO,15.860000,CAD,2651.459431,42052.145664,0.056070
4,LMT,565.500000,USD,42.357917,34318.756612,0.045759
5,AIG,79.680000,USD,353.700054,40378.371097,0.053839
6,AXP,301.700012,USD,83.899652,36266.014285,0.048355
7,ABBV,286.200012,USD,91.349370,37457.565514,0.049944
8,AAPL,342.559998,USD,80.493647,39505.957609,0.052675
9,UNP,287.290009,USD,104.943593,43195.727753,0.057595
10,ABT,103.000000,USD,244.906881,36141.198870,0.048189


In [37]:
#ranked = standard_deviation.sort_values()
#keep_share = 0.75
#num_of_shares = int(len(ranked) * keep_share)
#calm = ranked.index[:num_of_shares].tolist()
#print("Total eligible stocks:", len(ranked), "Kept in calm pool:", len(calm))

Total eligible stocks: 35 Kept in calm pool: 26


### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

Client B demands the minimum standard deviation of daily returns of his portfolio. And there are two aspects of determining portfolio risk: the risk of individual stocks and the relation between them (the formula for variance of two stocks has a correlation part, so lower correlation implies that more fluctuations compensate each other). We addressed both. We left in the portfolio only those 60% of the eligible stocks that had the minimum volatility. Secondly, from among that collection, the portfolio was created by choosing one security at a time, beginning with the least volatile, and then selecting the security having the least average correlation to the securities already selected, but no more than 7 for each sector so that no sector would dominate. That produced 18 stocks, and diversification means that with many stocks there is less risk associated with any particular company. Thirdly, each stock is weighted by one divided by its standard deviation so that calm stocks get allocated more dollars but, of course, subject to the 15% per stock and 40% per sector limitations.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

Our portfolio's standard deviation from Oct 1, 2025 through Sep 30, 2026, in CAD terms, was 0.686% per day (or 10.9% annualized), compared with 0.709% (annualized 11.3%) of the same 18 stocks with an equal weight, and earned about 12.1% vs. 10.2%. The 1 / standard deviation weighting thus reduced risk, albeit by a small amount, without reducing returns. Diversification is shown here not just because of this smaller standard deviation of 0.69%, relative to the typical stock in the universe of having a standard deviation of about 1% to 3%, but because these do not all move in sync, thereby offsetting each other. The numbers used here are based on the same history as the one used for the selection, and hence represent a conservative estimate.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

Correlations do not remain constant; during market sell-off periods, there is a likelihood that stocks will move down in unison, leading to higher correlation approaching 1 and reduced diversification. We base our portfolio choice entirely on historical information, and therefore, the stocks we identify as low-volatility may not exhibit such characteristic during the contest period; for example, news regarding the earnings release between November 30 and December 4 can result in an immediate one-day spike that overshadows volatility measured over five days. Low-volatility stocks are more likely to form part of defensive industries, and financials constitute the largest category in our portfolio (around one-third). The US stocks introduce currency risk since the contest performance is measured in Canadian Dollars.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

In [55]:
print('Period:', prices.index[0].date(), 'to', prices.index[-1].date(), '(all values in CAD)')
print('Client A')
display(comparison(weights_a))
print('Client B')
display(comparison(weights_b))

Period: 2025-10-01 to 2026-09-30 (all values in CAD)
Client A


,Total return,Daily std,Annualized std
Chosen weights,0.498428,0.014368,0.228085
Equal weights,0.422811,0.012070,0.191599


Client B


,Total return,Daily std,Annualized std
Chosen weights,0.120981,0.006864,0.108963
Equal weights,0.102328,0.007094,0.112607


## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

In [63]:
# runs the six rules from the assignment thenprints PASS or FAIL
def validate(table, fees, budget, name):
    # sector of each stock held, then total weight per sector
    sectors = info_stocks.loc[table['Ticker'], 'Sector'].values
    sector_weights = table.groupby(sectors)['Weight'].sum()

    
    # the 1e-9 allows for tinyrounding like this num: 0.15000000000001
    checks = {'Tickers are valid and eligible': set(table['Ticker']) <= set(filtered_tickers),
              'Between 10 and 20 stocks': 10 <= len(table) <= 20,
              'No stock above 15%': table['Weight'].max() <= 0.15 + 1e-9,
              'No sector above 40%': sector_weights.max() <= 0.40 + 1e-9,
              'Weights sum to 100%': abs(table['Weight'].sum() - 1) < 1e-9,
              'Purchases + fees within budget': table['Value'].sum() + fees <= budget}

    print(name)
    for check, passed in checks.items():
        print('  PASS' if passed else '  FAIL', check)

    #stop the notebook with an error if anything failed, so we wont getcooked by a bad performing portfolio
    assert all(checks.values())

validate(Portfolio_Final_A, fees_a, 1000000, 'Client A')
validate(Portfolio_Final_B, fees_b, 750000, 'Client B')

Client A
  PASS Tickers are valid and eligible
  PASS Between 10 and 20 stocks
  PASS No stock above 15%
  PASS No sector above 40%
  PASS Weights sum to 100%
  PASS Purchases + fees within budget
Client B
  PASS Tickers are valid and eligible
  PASS Between 10 and 20 stocks
  PASS No stock above 15%
  PASS No sector above 40%
  PASS Weights sum to 100%
  PASS Purchases + fees within budget


## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

In [57]:
print('Client A')
display(Portfolio_Final_A)
print('Client B')
display(Portfolio_Final_B)

reconciliation = pd.DataFrame({
    'Client A': [1000000, Portfolio_Final_A['Value'].sum(), fees_a, 1000000 - Portfolio_Final_A['Value'].sum() - fees_a],
    'Client B': [750000, Portfolio_Final_B['Value'].sum(), fees_b, 750000 - Portfolio_Final_B['Value'].sum() - fees_b]},
    index=['Budget', 'Purchases', 'Fees', 'Unspent'])
display(reconciliation)

Portfolio_Final_A[['Ticker', 'Shares']].to_csv('Stocks_Group_XX_A.csv', index=False)
Portfolio_Final_B[['Ticker', 'Shares']].to_csv('Stocks_Group_XX_B.csv', index=False)

Client A


,Ticker,Price,Currency,Shares,Value,Weight
1,CAT,510.450012,USD,204.769514,149898.733426,0.149900
2,BB.TO,11.100000,CAD,12740.090612,141415.010652,0.141416
3,TXN,263.750000,USD,305.716898,115635.546829,0.115637
4,QCOM,149.940002,USD,448.451858,96430.137615,0.096431
5,C,110.620003,USD,605.306563,96025.923699,0.096027
6,MRK,133.720001,USD,420.160446,80573.268660,0.080574
7,TD.TO,162.050003,CAD,432.902281,70151.815954,0.070152
8,AMZN,246.820007,USD,178.408474,63150.280800,0.063151
9,BIIB,249.589996,USD,136.493195,48855.967832,0.048856
10,RY.TO,263.049988,CAD,177.400128,46665.101440,0.046665


Client B


,Ticker,Price,Currency,Shares,Value,Weight
1,RY.TO,263.049988,CAD,247.781337,65178.877685,0.086906
2,MO,74.870003,USD,341.091333,36623.342472,0.048832
3,T.TO,15.860000,CAD,2651.459418,42052.145452,0.056070
4,LMT,565.500000,USD,42.317450,34318.756439,0.045759
5,AIG,79.680000,USD,353.362143,40378.370893,0.053839
6,AXP,301.700012,USD,83.819497,36266.014102,0.048355
7,ABBV,286.200012,USD,91.262099,37457.565325,0.049944
8,AAPL,342.559998,USD,80.416747,39505.957409,0.052675
9,UNP,287.290009,USD,104.843335,43195.727535,0.057595
10,ABT,103.000000,USD,244.672907,36141.198687,0.048189


,Client A,Client B
Budget,1000000.000000,750000.000000
Purchases,999991.550543,749990.500773
Fees,8.439457,9.489227
Unspent,0.010000,0.010000


## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):
Daniel Lei
Turki
Grace Xu
Ruoxi

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.